# Lesson 2 - Camera

The Jetson Nano reads the IMX219 CSI camera through GStreamer. `jbot.py` provides a
ready-made pipeline so OpenCV can grab frames.

We will capture a single frame, show a live stream with an interactive widget, and
plot the color histogram.

In [ ]:
import sys
sys.path.insert(0, "..")
%matplotlib inline
import cv2
import matplotlib.pyplot as plt
from jbot import snapshot, gst_pipeline

snapshot("frame.jpg")
img = cv2.imread("frame.jpg")[:, :, ::-1]
plt.figure(figsize=(8, 4))
plt.imshow(img)
plt.axis("off")
plt.title("Single frame")
plt.show()

## Focus the lens

The IMX219-160 lens is **manually focused**: you turn the small lens barrel by hand.
Run this cell, then slowly rotate the lens while watching the live image and the
**sharpness** bar. Stop when the bar is as high as possible (aim for > 150). Some
lenses have a tiny set screw or a dab of glue that must be loosened first, and a
protective film that must be peeled off.

In [ ]:
import threading
import time
import ipywidgets as widgets
from IPython.display import display

cap = cv2.VideoCapture(gst_pipeline(640, 360), cv2.CAP_GSTREAMER)
view = widgets.Image(format="jpeg", width=480, height=270)
meter = widgets.FloatProgress(value=0, min=0, max=300, description="sharp")
label = widgets.Label(value="turn the lens to maximize the bar")
stop = widgets.Button(description="Stop", button_style="danger")
display(widgets.VBox([view, meter, label, stop]))

state = {"run": True}

def loop():
    while state["run"]:
        ok, frame = cap.read()
        if ok:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            sharp = cv2.Laplacian(gray, cv2.CV_64F).var()
            meter.value = min(sharp, 300)
            label.value = "sharpness %.0f  (aim > 150)" % sharp
            view.value = cv2.imencode(".jpg", frame)[1].tobytes()
        time.sleep(0.05)
    cap.release()

stop.on_click(lambda _: state.update(run=False))
threading.Thread(target=loop, daemon=True).start()

## Live stream

This opens the camera and streams frames into an image widget. Press **Stop** to
release the camera. Only one process can hold the camera at a time, so stop the
stream before running other camera cells.

In [ ]:
import threading
import time
import ipywidgets as widgets
from IPython.display import display

cap = cv2.VideoCapture(gst_pipeline(640, 360), cv2.CAP_GSTREAMER)
view = widgets.Image(format="jpeg", width=640, height=360)
stop = widgets.Button(description="Stop", button_style="danger")
display(view, stop)

state = {"run": True}

def loop():
    while state["run"]:
        ok, frame = cap.read()
        if ok:
            view.value = cv2.imencode(".jpg", frame)[1].tobytes()
        time.sleep(0.03)
    cap.release()

stop.on_click(lambda _: state.update(run=False))
threading.Thread(target=loop, daemon=True).start()

## Color histogram

The histogram shows the distribution of blue, green and red pixel intensities in
the captured frame, a common first step in color-based vision.

In [ ]:
img = cv2.imread("frame.jpg")
plt.figure(figsize=(8, 3))
for i, c in enumerate(("b", "g", "r")):
    hist = cv2.calcHist([img], [i], None, [256], [0, 256])
    plt.plot(hist, color=c)
plt.xlim(0, 256)
plt.title("Color histogram")
plt.xlabel("intensity")
plt.ylabel("pixels")
plt.show()